## 0단계: 샘플 Excel 파일 생성 (최초 1회만)

In [ ]:
# 샘플 Excel 파일 생성
exec(open('create_kstartup_excel.py').read())

## 1단계: 환경 설정 및 드라이버 초기화

In [ ]:
import sys
from pathlib import Path
from datetime import datetime
import time
import logging

# 경로 설정
notebook_dir = Path.cwd()
sys.path.insert(0, str(notebook_dir))

# automation.py 임포트
from automation import WebAutomationConfig, WebActionParser, WebAutomationEngine

# 로깅 설정
logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s: %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('KStartup')

logger.info("✓ 환경 설정 완료")

In [ ]:
# WebDriver 초기화 (최초 1회만 실행)
config = WebAutomationConfig()
engine = WebAutomationEngine(config)
engine.initialize_driver()

# 드라이버를 전역 변수로 저장 (셀 간 공유)
driver = engine.driver

logger.info("✓ Chrome WebDriver 초기화 완료")
logger.info(f"현재 URL: {driver.current_url}")

## 2단계: Excel에서 계정 정보 및 액션 로드

In [ ]:
from openpyxl import load_workbook
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Excel 파일 경로 (파일명 수정 가능)
excel_path = "kstartup_actions.xlsx"

# Config 시트에서 인증 정보 로드
wb = load_workbook(excel_path)
ws_config = wb['Config']

credentials = {}
for row in ws_config.iter_rows(min_row=2, values_only=True):
    if row[0] and row[1]:
        credentials[row[0]] = row[1]

login_id = credentials.get('login_id', '')
login_pw = credentials.get('login_pw', '')
site_url = credentials.get('site_url', 'https://www.k-startup.go.kr/')

logger.info(f"✓ 계정 정보 로드: {login_id}")
logger.info(f"사이트 URL: {site_url}")

In [ ]:
# Actions 시트에서 작업 목록 로드
ws_actions = wb['Actions']

actions = []
for row in ws_actions.iter_rows(min_row=2, values_only=True):
    if row[0]:  # 순번이 있으면
        action = {
            'order': row[0],
            'name': row[1],
            'type': row[2],
            'xpath': row[3] or '',
            'value': row[4] or '',
            'wait': row[5] or 0,
            'desc': row[6] or ''
        }
        actions.append(action)

logger.info(f"✓ 작업 액션 로드: {len(actions)}개")

# 액션 목록 출력
for i, act in enumerate(actions[:5], 1):
    print(f"{i}. [{act['type']}] {act['name']}")
if len(actions) > 5:
    print(f"... 외 {len(actions)-5}개")

## 3단계: K-Startup 로그인

⚠️ **주의**: 실제 K-Startup 사이트의 로그인 요소를 확인하고 XPath를 수정하세요.
- F12 개발자 도구로 요소 검사
- 로그인 버튼, ID/PW 입력란의 실제 XPath 확인

In [ ]:
# K-Startup 메인 페이지 접속
driver.get(site_url)
time.sleep(2)

logger.info(f"✓ K-Startup 접속: {driver.current_url}")
logger.info(f"페이지 제목: {driver.title}")

In [ ]:
# 로그인 버튼 클릭 (XPath 수정 필요)
try:
    # 예시 XPath - 실제 사이트에 맞게 수정!
    login_btn_xpath = "//a[contains(text(), '로그인')]"
    login_btn = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, login_btn_xpath))
    )
    login_btn.click()
    time.sleep(2)
    logger.info("✓ 로그인 버튼 클릭")
except Exception as e:
    logger.error(f"로그인 버튼 찾기 실패: {e}")
    logger.info("수동으로 로그인 페이지로 이동하거나 XPath를 수정하세요.")

In [ ]:
# ID/PW 입력 및 로그인 (XPath 수정 필요)
try:
    # 예시 XPath - 실제 사이트에 맞게 수정!
    id_xpath = "//input[@name='userId' or @id='userId']"
    pw_xpath = "//input[@type='password']"
    submit_xpath = "//button[contains(text(), '로그인')]"
    
    # ID 입력
    id_field = driver.find_element(By.XPATH, id_xpath)
    id_field.clear()
    id_field.send_keys(login_id)
    logger.info(f"ID 입력: {login_id}")
    
    time.sleep(0.5)
    
    # PW 입력
    pw_field = driver.find_element(By.XPATH, pw_xpath)
    pw_field.clear()
    pw_field.send_keys(login_pw)
    logger.info("비밀번호 입력 완료")
    
    time.sleep(0.5)
    
    # 로그인 버튼 클릭
    submit_btn = driver.find_element(By.XPATH, submit_xpath)
    submit_btn.click()
    logger.info("✓ 로그인 시도")
    
    time.sleep(3)
    logger.info(f"현재 URL: {driver.current_url}")
    logger.info("✅ 로그인 완료 - 이제 세션이 유지됩니다!")
    
except Exception as e:
    logger.error(f"로그인 실패: {e}")
    logger.info("XPath를 확인하거나 수동으로 로그인하세요.")

## 4단계: 작업 수행 (원하는 셀만 실행)

여기서부터는 로그인된 상태에서 원하는 작업만 반복 실행할 수 있습니다.

In [ ]:
# 현재 페이지 정보 확인
print(f"현재 URL: {driver.current_url}")
print(f"페이지 제목: {driver.title}")

In [ ]:
# 특정 작업 실행 - 순번으로 선택
# 예: 순번 5~10번 작업만 실행

start_order = 5  # 시작 순번
end_order = 10   # 종료 순번

selected_actions = [a for a in actions if start_order <= a['order'] <= end_order]

logger.info(f"순번 {start_order}~{end_order} 작업 실행: {len(selected_actions)}개")

for action in selected_actions:
    try:
        action_type = str(action['type']).lower()
        
        if action_type == 'navigate':
            driver.get(action['value'])
            logger.info(f"[{action['order']}] 페이지 이동: {action['value']}")
        
        elif action_type == 'wait':
            time.sleep(float(action['value']))
            logger.info(f"[{action['order']}] 대기: {action['value']}초")
        
        elif action_type == 'click':
            element = WebDriverWait(driver, 10).until(
                EC.element_to_be_clickable((By.XPATH, action['xpath']))
            )
            element.click()
            logger.info(f"[{action['order']}] 클릭: {action['name']}")
        
        elif action_type == 'input':
            element = driver.find_element(By.XPATH, action['xpath'])
            element.clear()
            element.send_keys(action['value'])
            logger.info(f"[{action['order']}] 입력: {action['name']}")
        
        elif action_type == 'get_text':
            element = driver.find_element(By.XPATH, action['xpath'])
            text = element.text
            logger.info(f"[{action['order']}] 텍스트: {text}")
        
        else:
            logger.warning(f"[{action['order']}] 지원하지 않는 액션: {action_type}")
        
        # 액션 간 짧은 대기
        if action.get('wait'):
            time.sleep(float(action['wait']))
    
    except Exception as e:
        logger.error(f"[{action['order']}] 실패: {action['name']} - {e}")

logger.info("✓ 선택한 작업 완료")

In [ ]:
# 특정 요소 찾기 테스트
# XPath를 입력하고 요소가 있는지 확인

test_xpath = "//div[@class='some-class']"  # 테스트할 XPath

try:
    element = driver.find_element(By.XPATH, test_xpath)
    print(f"✓ 요소 찾음")
    print(f"텍스트: {element.text}")
    print(f"HTML: {element.get_attribute('outerHTML')[:200]}")
except Exception as e:
    print(f"✗ 요소를 찾지 못함: {e}")

In [ ]:
# 페이지 소스 일부 확인 (디버깅용)
print("페이지 HTML (처음 1000자):")
print(driver.page_source[:1000])

## 5단계: 브라우저 종료

작업이 완전히 끝났을 때만 실행하세요.

In [ ]:
# 브라우저 종료
driver.quit()
logger.info("✓ 브라우저 종료")

---

## 사용 팁

### 개발 워크플로우
1. **1~3단계 실행**: 환경 설정 → Excel 로드 → 로그인
2. **4단계에서 반복 테스트**: 
   - `start_order`, `end_order`를 조정하여 원하는 구간만 실행
   - 각 셀을 개별적으로 실행하며 테스트
3. **XPath 수정**: 실제 사이트에 맞게 XPath를 조정
4. **Excel 업데이트**: 작업이 완성되면 Excel에 저장

### 배포 (자동화 실행)
개발이 완료되면 `automation.py`를 사용하여 자동 실행:
```bash
# 전체 실행
python automation.py kstartup_actions.xlsx

# 구간 실행
python automation.py kstartup_actions.xlsx --start 5 --end 10
```

### 주의사항
- K-Startup 사이트 구조에 따라 XPath를 수정해야 합니다
- 로그인 정보는 Config 시트에서 관리합니다
- 브라우저를 종료하지 않으면 세션이 계속 유지됩니다